In [0]:
from pyspark.sql import functions as F, Window as W

cat = "capstone6_telecom.telecom"
EXTREME_LATENCY_MS = 500                   
EVENT_STATUS = ["OPEN", "FAILED", "PENDING", "COMPLETED"]
CDC_STATUS   = EVENT_STATUS + ["CANCELLED", "REVERSED"]
SEC_TYPES    = ["CHECK", "ALERT", "REVIEW"]
OUTAGE_STATUS = "FAILED"                

In [0]:
def failed_rules(*pairs):
    """pairs = (condition_meaning_the_rule_FAILED, rule_id). Returns an array of failed rule ids."""
    return F.filter(F.array(*[F.when(c, F.lit(r)) for c, r in pairs]), lambda x: x.isNotNull())

def not_in(col, values):
    """True when the value is null or not in the list."""
    return F.coalesce(~F.col(col).isin(values), F.lit(True))

def rejected_rows(df, record_type, key_col, raw_cols):
    """df needs: rule_id, action, source_file, ingest_ts, key_col and raw_cols."""
    rules = spark.table(f"{cat}.silver_dq_rules").select("rule_id", F.col("description").alias("reason"))
    return (df.join(F.broadcast(rules), "rule_id", "left")
              .select(F.lit(record_type).alias("record_type"),
                      F.col(key_col).cast("string").alias("record_key"),
                      "rule_id", "action", "reason", "source_file",
                      F.col("ingest_ts").alias("bronze_ingest_ts"),
                      F.to_json(F.struct(*raw_cols)).alias("raw_record"),
                      F.current_timestamp().alias("rejected_at")))

def write_rejected(df, record_type):
    """Idempotent: replaces only this record_type's rows, so re-runs never duplicate."""
    t = f"{cat}.silver_rejected_records"
    if spark.catalog.tableExists(t):
        (df.write.format("delta").mode("overwrite")
           .option("replaceWhere", f"record_type = '{record_type}'").saveAsTable(t))
    else:
        df.write.format("delta").saveAsTable(t)

In [0]:
def build_silver_events(b, regions, towers):
    """Bronze events -> (clean, rejected). Keeps raw values in Bronze; types and flags here."""
    if "source_schema_version" not in b.columns:
        b = b.withColumn("source_schema_version", F.lit(None).cast("string"))
    raw_cols = b.columns

    t = (b
         .withColumn("event_ts", F.expr("try_to_timestamp(event_timestamp)"))
         .withColumn("source_ingest_ts", F.expr("try_to_timestamp(ingest_timestamp)"))
         .withColumn("latency_n", F.expr("try_cast(latency_ms AS DOUBLE)"))
         .withColumn("loss_n", F.expr("try_cast(packet_loss_pct AS DOUBLE)"))
         .withColumn("status_c", F.upper(F.trim(F.col("network_status"))))
         .withColumn("type_c", F.expr("nullif(upper(trim(network_type)), '')")))

    # Deterministic dedup: newest Bronze load wins, then highest file name.
    w = W.partitionBy("measurement_id").orderBy(F.col("ingest_ts").desc(), F.col("source_file").desc())
    t = (t.withColumn("rn", F.row_number().over(w))
          .withColumn("copies", F.count("*").over(W.partitionBy("measurement_id")))
          .withColumn("is_dup", F.col("measurement_id").isNotNull() & (F.col("rn") > 1)))

    t = (t.join(F.broadcast(regions.select("region_id").distinct().withColumn("_reg_ok", F.lit(True))), "region_id", "left")
          .join(F.broadcast(towers.select("tower_id").distinct().withColumn("_twr_ok", F.lit(True))), "tower_id", "left"))

    t = t.withColumn("failed", failed_rules(
        (F.col("measurement_id").isNull() | (F.trim(F.col("measurement_id")) == ""), "DQ01"),
        (F.col("event_ts").isNull(), "DQ02"),
        (F.col("latency_n").isNull() | (F.col("latency_n") < 0), "DQ03"),
        (F.col("loss_n").isNull() | (F.col("loss_n") < 0) | (F.col("loss_n") > 100), "DQ04"),
        (not_in("status_c", EVENT_STATUS), "DQ05"),
        (F.col("_reg_ok").isNull(), "DQ06"),
        (F.col("_twr_ok").isNull(), "DQ07")))

    clean = (t.filter(~F.col("is_dup") & (F.size("failed") == 0))
              .select("measurement_id", "event_ts", "source_ingest_ts", "region_id", "tower_id",
                      F.coalesce("type_c", F.lit("UNKNOWN")).alias("network_type"),
                      F.col("type_c").isNull().alias("network_type_missing"),          # DQ09 flag
                      F.col("status_c").alias("network_status"), "unit",
                      F.col("latency_n").alias("latency_ms"),
                      F.col("loss_n").alias("packet_loss_pct"),
                      (F.col("latency_n") > EXTREME_LATENCY_MS).alias("latency_outlier_flag"),  # DQ10 flag
                      "operator_id", "source_system", "source_schema_version",
                      (F.col("copies") - 1).alias("duplicate_copies_dropped"),
                      "source_file", F.col("ingest_ts").alias("bronze_ingest_ts")))

    rej_fail = (t.filter(~F.col("is_dup") & (F.size("failed") > 0))
                 .withColumn("rule_id", F.explode("failed")).withColumn("action", F.lit("REJECTED")))
    rej_dup = (t.filter(F.col("is_dup"))
                .withColumn("rule_id", F.lit("DQ08")).withColumn("action", F.lit("DEDUPED")))
    rejected = rejected_rows(rej_fail.unionByName(rej_dup), "EVENT", "measurement_id", raw_cols)
    return clean, rejected

## 1. DQ rule catalogue (rule id, severity, action)

In [0]:
rules = [
 ("DQ01","network_events","measurement_id is not null or blank","ERROR","REJECT"),
 ("DQ02","network_events","event_timestamp must parse as a timestamp","ERROR","REJECT"),
 ("DQ03","network_events","latency_ms must be numeric and >= 0","ERROR","REJECT"),
 ("DQ04","network_events","packet_loss_pct must be numeric between 0 and 100","ERROR","REJECT"),
 ("DQ05","network_events","network_status must be OPEN, FAILED, PENDING or COMPLETED","ERROR","REJECT"),
 ("DQ06","network_events","region_id must exist in the region reference table","ERROR","REJECT"),
 ("DQ07","network_events","tower_id must exist in the tower reference table","ERROR","REJECT"),
 ("DQ08","network_events","exact duplicate measurement_id (older copy dropped)","INFO","DEDUPE"),
 ("DQ09","network_events","network_type missing (kept, set to UNKNOWN, flagged)","WARN","FLAG"),
 ("DQ10","network_events","latency_ms above outlier threshold (kept, flagged)","WARN","FLAG"),
 ("DQ11","status_cdc","change refers to a measurement_id that is not in silver events","ERROR","REJECT"),
 ("DQ12","status_cdc","new_status is not a known status","ERROR","REJECT"),
 ("DQ13","status_cdc","change_timestamp missing, unparseable, or earlier than the event","ERROR","REJECT"),
 ("DQ14","status_cdc","source_sequence missing or not a number","ERROR","REJECT"),
 ("DQ15","status_cdc","duplicate (measurement_id, source_sequence) replay","INFO","DEDUPE"),
 ("DQ16","secondary_events","event_id missing, or duplicate event_id (older copy dropped)","ERROR","REJECT/DEDUPE"),
 ("DQ17","secondary_events","measurement_id not found in silver events","ERROR","REJECT"),
 ("DQ18","secondary_events","event_type must be CHECK, ALERT or REVIEW","ERROR","REJECT"),
 ("DQ19","secondary_events","event_value must be numeric","ERROR","REJECT"),
 ("DQ20","secondary_events","event_timestamp must parse as a timestamp","ERROR","REJECT"),
 ("DQ21","secondary_events","related_entity_id must exist in the region reference table","ERROR","REJECT"),
]
(spark.createDataFrame(rules, "rule_id string, dataset string, description string, severity string, action string")
      .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(f"{cat}.silver_dq_rules"))
display(spark.table(f"{cat}.silver_dq_rules"))

## 2. Reference dimensions (typed copies of Bronze)

In [0]:
def make_dim(src, key):
    return (spark.table(f"{cat}.{src}")
            .select(key, "display_name", F.col("region").alias("region_group"),
                    (F.upper(F.trim("active_flag")) == "Y").alias("is_active"),
                    F.expr("try_cast(effective_date AS DATE)").alias("effective_date"),
                    "risk_tier")
            .dropDuplicates([key]))

make_dim("region_id", "region_id").write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{cat}.silver_dim_region")
make_dim("tower_id",  "tower_id").write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{cat}.silver_dim_tower")
print(spark.table(f"{cat}.silver_dim_region").count(), "regions (expect 110);",
      spark.table(f"{cat}.silver_dim_tower").count(), "towers (expect 150)")

## 3. Build silver_network_events and the rejected records

In [0]:
bronze  = spark.table(f"{cat}.bronze_network_events")
regions = spark.table(f"{cat}.silver_dim_region")
towers  = spark.table(f"{cat}.silver_dim_tower")

clean, rejected = build_silver_events(bronze, regions, towers)

clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{cat}.silver_network_events")
write_rejected(rejected, "EVENT")

## 4. Before / after counts (evidence)
Expected: Bronze 11,540 = Silver 11,500 + 40 deduped + 0 rejected.

In [0]:
display(spark.sql(f"""
SELECT (SELECT COUNT(*) FROM {cat}.bronze_network_events) AS bronze_rows,
       (SELECT COUNT(*) FROM {cat}.silver_network_events) AS silver_rows,
       (SELECT COUNT(*) FROM {cat}.silver_rejected_records WHERE record_type='EVENT' AND action='DEDUPED')  AS deduped,
       (SELECT COUNT(*) FROM {cat}.silver_rejected_records WHERE record_type='EVENT' AND action='REJECTED') AS rejected"""))

display(spark.sql(f"""
SELECT SUM(CAST(network_type_missing AS INT)) AS network_type_unknown,        -- expect 12
       SUM(CAST(latency_outlier_flag AS INT)) AS latency_outliers_flagged,    -- expect 35
       COUNT(*) - COUNT(DISTINCT measurement_id) AS duplicate_ids_left        -- expect 0
FROM {cat}.silver_network_events"""))

display(spark.sql(f"""SELECT rule_id, action, COUNT(*) AS rows
FROM {cat}.silver_rejected_records WHERE record_type='EVENT' GROUP BY 1,2 ORDER BY 1"""))